Análisis Descriptivo - Accidentalidad Caldas
Dataset de accidentes de tránsito con análisis estadístico descriptivo:
- Visualizaciones de distribuciones
- Análisis bivariado: Chi-cuadrado y t de Student
- Correlación entre variables numéricas

## 1. Importar librerías

In [ ]:
import numpy as np                        # Operaciones matemáticas con vectores y matrices
import pandas as pd                       # Manejo de DataFrames (tablas de datos)
import seaborn as sns                     # Visualización estadística
import matplotlib.pyplot as plt           # Gráficos base

from scipy.stats import chi2_contingency  # Prueba Chi-cuadrado (categórica vs categórica)
from scipy.stats import ttest_ind         # Prueba t de Student (numérica vs categórica)

import warnings
warnings.filterwarnings('ignore')

## 2. Cargar dataset limpio

In [ ]:
df = pd.read_csv('../datasets_limpios/accidentalidad_caldas_limpio.csv', sep=';', encoding='utf-8')

print('Dimensiones:', df.shape)
display(df.head(10))

## 3. Gráficas de distribución

In [ ]:
orden_dias = ['LUNES','MARTES','MIERCOLES','JUEVES','VIERNES','SABADO','DOMINGO']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Distribución de GRAVEDAD
sns.countplot(data=df, x='GRAVEDAD', palette='Set2', ax=axes[0,0],
              order=df['GRAVEDAD'].value_counts().index)
axes[0,0].set_title('Accidentes por Gravedad')
axes[0,0].set_xlabel('')

# Gráfico 2: Distribución de CLASE
sns.countplot(data=df, x='CLASE', palette='Set1', ax=axes[0,1],
              order=df['CLASE'].value_counts().index)
axes[0,1].set_title('Accidentes por Clase')
axes[0,1].set_xlabel('')
axes[0,1].tick_params(axis='x', rotation=20)

# Gráfico 3: Accidentes por Franja Horaria
sns.countplot(data=df, x='FRANJA_HORARIA', palette='Blues_d', ax=axes[1,0],
              order=['Madrugada (0-5)','Mañana (6-11)','Tarde (12-17)','Noche (18-23)'])
axes[1,0].set_title('Accidentes por Franja Horaria')
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=15)

# Gráfico 4: Accidentes por Día de la Semana
sns.countplot(data=df, x='DÍA DE LA SEMANA', palette='Oranges', ax=axes[1,1],
              order=orden_dias)
axes[1,1].set_title('Accidentes por Día de la Semana')
axes[1,1].set_xlabel('')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Verifica el conteo exacto por día
print(df['DÍA DE LA SEMANA'].value_counts())

## 4. Análisis Bivariado: Chi-cuadrado (Categórica vs Categórica)
Usamos Chi-cuadrado para saber si dos variables categóricas están relacionadas.

- H₀ (hipótesis nula):las variables son independientes (no hay relación)
- Si el p-valor < 0.05 → rechazamos H₀ → sí hay relación estadística

Aplicamos a:CLASE vs GRAVEDAD,DISEÑO vs GRAVEDAD,DÍA vs GRAVEDAD

In [ ]:
def chi_cuadrado(df, var1, var2):
    """
    Realiza la prueba Chi-cuadrado entre dos variables categóricas.
    Muestra la tabla de contingencia y el resultado del test.
    """
    # Tabla de contingencia: cruza los valores de var1 con var2 y cuenta ocurrencias
    tabla = pd.crosstab(df[var1], df[var2])
    print(f'Tabla de contingencia: {var1} vs {var2}')
    display(tabla)

    # chi2_contingency aplica la prueba estadística sobre la tabla
    # Devuelve: estadístico chi2, p-valor, grados de libertad, frecuencias esperadas
    chi2, p, dof, expected = chi2_contingency(tabla)

    print(f'Chi2 = {chi2:.4f} | p-valor = {p:.4f} | Grados de libertad = {dof}')
    if p < 0.05:
        print(f'Resultado: SÍ existe relación estadística entre {var1} y {var2} (p < 0.05)\n')
    else:
        print(f'Resultado: NO hay relación estadística entre {var1} y {var2} (p >= 0.05)\n')

# Aplicamos la prueba a los tres pares de variables
chi_cuadrado(df, 'CLASE',           'GRAVEDAD')
chi_cuadrado(df, 'DISEÑO',          'GRAVEDAD')
chi_cuadrado(df, 'DÍA DE LA SEMANA','GRAVEDAD')

## 5. Análisis Bivariado: t de Student (Numérica vs Categórica)
Usamos t de Student para comparar la media de una variable numérica entre dos grupos.

Pregunta: ¿La hora del accidente es diferente entre accidentes con DAÑOS vs HERIDOS?

- Si p-valor < 0.05 → las medias son significativamente diferentes → la hora sí influye en la gravedad

In [ ]:
# Separamos los dos grupos: accidentes con DAÑOS y accidentes con HERIDOS
grupo_danos   = df[df['GRAVEDAD'] == 'DAÑOS'  ]['HORA_NUM'].dropna()
grupo_heridos = df[df['GRAVEDAD'] == 'HERIDOS']['HORA_NUM'].dropna()

print(f'Media hora - DAÑOS:   {grupo_danos.mean():.2f}')
print(f'Media hora - HERIDOS: {grupo_heridos.mean():.2f}')
print()

# ttest_ind compara las medias de dos grupos independientes
# Devuelve el estadístico t y el p-valor
t_stat, p_valor = ttest_ind(grupo_danos, grupo_heridos)

print(f'Estadístico t = {t_stat:.4f} | p-valor = {p_valor:.4f}')
if p_valor < 0.05:
    print(' Resultado: La hora del accidente SÍ difiere significativamente entre DAÑOS y HERIDOS')
else:
    print(' Resultado: La hora del accidente NO difiere significativamente entre DAÑOS y HERIDOS')

# Visualización: boxplot para comparar las distribuciones
plt.figure(figsize=(7, 5))
subset = df[df['GRAVEDAD'].isin(['DAÑOS', 'HERIDOS'])]
sns.boxplot(data=subset, x='GRAVEDAD', y='HORA_NUM', palette='pastel')
plt.title('Distribución de la hora según gravedad del accidente')
plt.xlabel('Gravedad')
plt.ylabel('Hora del día (0–23)')
plt.tight_layout()
plt.show()

## 6. Mapa de calor — Correlación entre variables numéricas
Muestra qué tan relacionadas están las variables numéricas entre sí.
Valores cercanos a 1 o -1 indican mayor correlación.

In [ ]:
# Seleccionamos las columnas numéricas relevantes
cols_num = ['AÑO', 'MES', 'DIA_MES', 'HORA_NUM', 'ES_FIN_SEMANA',
            'DIA_cod', 'GRAVEDAD_cod']

correlacion = df[cols_num].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Mapa de calor — Correlación entre variables numéricas')
plt.tight_layout()
plt.show()